In [1]:
# Paths and options (relative to the project root)
PROJECT_ROOT_OVERRIDE = ""   # Set only if the notebook is outside the project directory.
OUTPUT_RELATIVE_DIR = "outputs/asama18_verified_v13"
SEEDS = (42, 1, 7)
MODELS = ("convnext_tiny", "resnet50", "retfound_fullft")
DATASETS = ("WBCBench", "PBC", "Raabin_Train_TestA")
EXPORT_XAI_FIGURE4 = True

In [2]:
from pathlib import Path
import csv, json, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score
from IPython.display import display

MODEL_NAMES = {"convnext_tiny": "ConvNeXt-Tiny", "resnet50": "ResNet-50", "retfound_fullft": "RETFound"}
DATASET_NAMES = {"WBCBench": "WBCBench", "PBC": "PBC", "Raabin_Train_TestA": "Raabin-WBC"}
CLASSES_13 = ("BA", "BL", "BNE", "EO", "LY", "MMY", "MO", "MY", "PC", "PLY", "PMY", "SNE", "VLY")
CLASSES_5 = ("BA", "EO", "LY", "MO", "SNE")
EXPECTED_SIZE = {"WBCBench": 32934, "PBC": 10298, "Raabin_Train_TestA": 14514}

start = Path(PROJECT_ROOT_OVERRIDE).expanduser() if PROJECT_ROOT_OVERRIDE else Path.cwd()
possible = [start, *start.parents]
R = next((p for p in possible if (p / "review_results_external_consistency_v13" / "external_consistency_decision_v13.json").is_file()
          and (p / "outputs" / "final_test_v2").is_dir()), None)
if R is None:
    raise FileNotFoundError("Project root not found. Open the notebook from sci5-wbc, or set PROJECT_ROOT_OVERRIDE.")
R = R.resolve()
V13 = R / "review_results_external_consistency_v13"
V10 = R / "review_results_ret_found_only_v10"
V6 = R / "review_results_external_BNE_SNE_v6"
WBC = R / "outputs" / "final_test_v2"
O = R / OUTPUT_RELATIVE_DIR
O.mkdir(parents=True, exist_ok=True)
print("Project root:", R)
print("Export directory:", O)

Project root: C:\Users\sarah\Desktop\scı5-wbc
Export directory: C:\Users\sarah\Desktop\scı5-wbc\outputs\asama18_verified_v13


In [3]:
# Check V13 provenance before producing any output.
with (V13 / "external_consistency_decision_v13.json").open(encoding="utf-8") as stream:
    decision = json.load(stream)
assert decision.get("all_18_cached_runs_verified") is True, "V13 did not verify all external runs."
assert int(decision.get("verified_runs", -1)) == 18, "Expected 18 verified runs."
assert int(decision.get("issues_count", -1)) == 0, "V13 reported audit issues."
assert decision.get("external_ci_recomputed") is True, "V13 paired confidence intervals are missing."

per_seed = pd.read_csv(V13 / "external_per_seed_verified_v13.csv")
external = pd.read_csv(V13 / "external_model_summary_verified_v13.csv")
ci = pd.read_csv(V13 / "external_corrected_seed_average_bootstrap_v13.csv")
assert len(per_seed) == 18 and per_seed["status"].eq("PASS_CACHED_EXTERNAL").all()
assert len(external) == 6 and len(ci) == 6
expected_pairs = {(d, m, s) for d in DATASETS[1:] for m in MODELS for s in SEEDS}
assert set(zip(per_seed.dataset, per_seed.model, per_seed.seed)) == expected_pairs
assert set(zip(external.dataset, external.model)) == {(d, m) for d in DATASETS[1:] for m in MODELS}
assert (external.n_seeds == 3).all()
assert set(ci.bootstrap_n.astype(int)) == {1500}
assert ci.resampling.eq("class_stratified_paired_image").all()
assert ci.estimator.eq("mean_over_3_seed_macro_f1_difference").all()
for (d, m), g in per_seed.groupby(["dataset", "model"]):
    s = external[(external.dataset == d) & (external.model == m)].iloc[0]
    assert len(g) == 3 and g.n.eq(EXPECTED_SIZE[d]).all()
    assert np.isclose(g.baseline_macro_f1.mean(), s.baseline_mean, atol=1e-9)
    assert np.isclose(g.baseline_macro_f1.std(ddof=1), s.baseline_sample_sd, atol=1e-9)
    assert np.isclose(g.bne_sne_macro_f1.mean(), s.bne_sne_mean, atol=1e-9)

print("Verified V13 external records: 18 / 18; paired bootstrap comparisons: 6 / 6")
print("WBCBench V12 independent replay status:", decision.get("wbcbench_v12_full_verification", "not supplied"))

Verified V13 external records: 18 / 18; paired bootstrap comparisons: 6 / 6
WBCBench V12 independent replay status: REVIEW__FULL_NOT_COMPLETELY_VALIDATED


In [4]:
# Load the retained WBCBench 13-class test predictions (no inference required).
def load_wbc(seed, model):
    folder = WBC / model / f"seed{seed}"
    if not folder.is_dir():
        raise FileNotFoundError(f"Missing retained WBCBench test predictions: {folder}")
    y = np.load(folder / "true.npy", allow_pickle=False)
    p = np.load(folder / "pred.npy", allow_pickle=False)
    if y.shape != (EXPECTED_SIZE["WBCBench"],) or p.shape != y.shape:
        raise ValueError(f"Unexpected WBCBench shape: {model}, {seed}: {y.shape}, {p.shape}")
    if not np.isin(y, np.arange(13)).all() or not np.isin(p, np.arange(13)).all():
        raise ValueError("WBCBench class IDs must range from 0 to 12.")
    meta = folder / "DONE.json"
    if meta.is_file():
        recorded = json.loads(meta.read_text(encoding="utf-8"))
        if "macro_f1" in recorded:
            recomputed = f1_score(y, p, labels=np.arange(13), average="macro", zero_division=0)
            if not np.isclose(recomputed, float(recorded["macro_f1"]), atol=1e-6):
                raise ValueError(f"WBCBench DONE.json score mismatch: {model}, seed{seed}")
    return y, p

wbc_arrays = {(m, s): load_wbc(s, m) for m in MODELS for s in SEEDS}
reference = wbc_arrays[(MODELS[0], SEEDS[0])][0]
assert all(np.array_equal(y, reference) for y, _ in wbc_arrays.values()), "WBCBench labels/order differ across models or seeds."
print("WBCBench retained records checked: 9 / 9", "| n =", len(reference))

WBCBench retained records checked: 9 / 9 | n = 32934


In [5]:
# Cross-check external cached 5-class predictions against the 18 V13 seed-level metrics.
def external_folder(dataset, model, seed):
    if model == "retfound_fullft":
        return V10 / dataset / f"seed{seed}"
    return V6 / dataset / model / f"seed{seed}"

def load_external(dataset, model, seed):
    folder = external_folder(dataset, model, seed)
    y = np.load(folder / "true_5class.npy", allow_pickle=False)
    p = np.load(folder / "pred_baseline_5class.npy", allow_pickle=False)
    if y.shape != (EXPECTED_SIZE[dataset],) or p.shape != y.shape:
        raise ValueError(f"Unexpected external shape: {dataset}, {model}, seed{seed}")
    if not np.isin(y, np.arange(5)).all() or not np.isin(p, np.arange(5)).all():
        raise ValueError("External class IDs must range from 0 to 4.")
    csv_row = per_seed[(per_seed.dataset == dataset) & (per_seed.model == model) & (per_seed.seed == seed)].iloc[0]
    score = f1_score(y, p, labels=np.arange(5), average="macro", zero_division=0)
    if not np.isclose(score, csv_row.baseline_macro_f1, atol=1e-8):
        raise ValueError(f"External score mismatch: {dataset}, {model}, seed{seed}")
    return y, p

ext_arrays = {(d, m, s): load_external(d, m, s) for d in DATASETS[1:] for m in MODELS for s in SEEDS}
for d in DATASETS[1:]:
    reference = ext_arrays[(d, MODELS[0], SEEDS[0])][0]
    assert all(np.array_equal(ext_arrays[d, m, s][0], reference) for m in MODELS for s in SEEDS), f"{d}: label/order mismatch"
print("External cached prediction files cross-checked: 18 / 18")

External cached prediction files cross-checked: 18 / 18


In [6]:
# Table 1 — three-seed performance. Calculate ALL three metrics from the corrected predictions.
rows = []
for d in DATASETS:
    for m in MODELS:
        for s in SEEDS:
            y, p = wbc_arrays[(m,s)] if d == "WBCBench" else ext_arrays[(d,m,s)]
            labels = np.arange(13 if d == "WBCBench" else 5)
            rows.append({"dataset": d, "model": m, "seed": s, "n_images": len(y),
                         "macro_f1": f1_score(y, p, labels=labels, average="macro", zero_division=0),
                         "balanced_accuracy": balanced_accuracy_score(y, p),
                         "accuracy": accuracy_score(y, p)})
metrics = pd.DataFrame(rows)

agg = metrics.groupby(["dataset", "model"], sort=False).agg(
    n_images=("n_images", "first"), n_seeds=("seed", "nunique"),
    macro_f1_mean=("macro_f1", "mean"), macro_f1_sd=("macro_f1", "std"),
    balanced_accuracy_mean=("balanced_accuracy", "mean"), balanced_accuracy_sd=("balanced_accuracy", "std"),
    accuracy_mean=("accuracy", "mean"), accuracy_sd=("accuracy", "std")
).reset_index()
assert len(metrics) == 27 and len(agg) == 9
for _, e in external.iterrows():
    a = agg[(agg.dataset == e.dataset) & (agg.model == e.model)].iloc[0]
    assert np.isclose(a.macro_f1_mean, e.baseline_mean, atol=1e-8)
    assert np.isclose(a.macro_f1_sd, e.baseline_sample_sd, atol=1e-8)

metrics.to_csv(O / "Table1_PerSeed_Performance.csv", index=False, encoding="utf-8-sig")
agg.to_csv(O / "Table1_ThreeSeed_Performance_Numeric.csv", index=False, encoding="utf-8-sig")
formatted = pd.DataFrame([{
    "Dataset": DATASET_NAMES[r.dataset], "Task": "13-class" if r.dataset == "WBCBench" else "5-class",
    "Model": MODEL_NAMES[r.model], "n": int(r.n_images),
    "Macro-F1 (mean ± SD)": f"{r.macro_f1_mean:.4f} ± {r.macro_f1_sd:.4f}",
    "Balanced accuracy (mean ± SD)": f"{r.balanced_accuracy_mean:.4f} ± {r.balanced_accuracy_sd:.4f}",
    "Accuracy (mean ± SD)": f"{r.accuracy_mean:.4f} ± {r.accuracy_sd:.4f}"
} for r in agg.itertuples()])
formatted.to_csv(O / "Table1_Main_Performance.csv", index=False, encoding="utf-8-sig")
display(formatted)

,Dataset,Task,Model,n,Macro-F1 (mean ± SD),Balanced accuracy (mean ± SD),Accuracy (mean ± SD)
0,WBCBench,13-class,ConvNeXt-Tiny,32934,0.6367 ± 0.0085,0.6254 ± 0.0233,0.9494 ± 0.0043
1,WBCBench,13-class,ResNet-50,32934,0.6466 ± 0.0143,0.6353 ± 0.0181,0.9486 ± 0.0041
2,WBCBench,13-class,RETFound,32934,0.6107 ± 0.0052,0.5977 ± 0.0019,0.9407 ± 0.0009
3,PBC,5-class,ConvNeXt-Tiny,10298,0.9650 ± 0.0114,0.9620 ± 0.0099,0.9720 ± 0.0088
4,PBC,5-class,ResNet-50,10298,0.9255 ± 0.0160,0.9197 ± 0.0158,0.9425 ± 0.0111
5,PBC,5-class,RETFound,10298,0.8734 ± 0.0072,0.8636 ± 0.0137,0.8919 ± 0.0141
6,Raabin-WBC,5-class,ConvNeXt-Tiny,14514,0.7076 ± 0.0747,0.7098 ± 0.0123,0.8968 ± 0.0377
7,Raabin-WBC,5-class,ResNet-50,14514,0.6209 ± 0.0539,0.6167 ± 0.0254,0.8519 ± 0.0273
8,Raabin-WBC,5-class,RETFound,14514,0.4408 ± 0.0250,0.4572 ± 0.0265,0.8211 ± 0.0215


In [7]:
# Table 2 — WBCBench 13-class per-class F1, calculated from locked test predictions.
per_class_rows = []
for m in MODELS:
    for s in SEEDS:
        y,p = wbc_arrays[(m,s)]
        f1 = f1_score(y, p, labels=np.arange(13), average=None, zero_division=0)
        for name, v in zip(CLASSES_13, f1):
            per_class_rows.append({"model":m,"seed":s,"class":name,"f1":float(v),
                                   "true_support": int(np.count_nonzero(y == CLASSES_13.index(name)))})
pc_df = pd.DataFrame(per_class_rows)
pc_stats = pc_df.groupby(["class","model"], sort=False).agg(
    mean_f1=("f1", "mean"), sample_sd=("f1", "std"), support=("true_support", "first")
).reset_index()
result = []
for cname in CLASSES_13:
    v = {"Class": cname, "Support": int(pc_stats[pc_stats["class"] == cname]["support"].iloc[0])}
    for m in MODELS:
        z = pc_stats[(pc_stats["class"] == cname) & (pc_stats["model"] == m)].iloc[0]
        v[MODEL_NAMES[m]] = f"{z.mean_f1:.3f} ± {z.sample_sd:.3f}"
    result.append(v)
per_class_table = pd.DataFrame(result)
per_class_table.to_csv(O / "Table2_WBCBench_PerClass_F1.csv", index=False, encoding="utf-8-sig")
display(per_class_table)

,Class,Support,ConvNeXt-Tiny,ResNet-50,RETFound
0,BA,606,0.949 ± 0.014,0.954 ± 0.006,0.948 ± 0.006
1,BL,1770,0.882 ± 0.018,0.893 ± 0.007,0.877 ± 0.008
2,BNE,252,0.388 ± 0.055,0.264 ± 0.111,0.240 ± 0.005
3,EO,984,0.961 ± 0.005,0.960 ± 0.006,0.956 ± 0.007
4,LY,8598,0.964 ± 0.003,0.964 ± 0.005,0.958 ± 0.002
5,MMY,340,0.505 ± 0.029,0.513 ± 0.008,0.404 ± 0.059
6,MO,3066,0.936 ± 0.007,0.929 ± 0.010,0.909 ± 0.005
7,MY,408,0.716 ± 0.050,0.754 ± 0.004,0.589 ± 0.050
8,PC,24,0.000 ± 0.000,0.027 ± 0.047,0.000 ± 0.000
9,PLY,4,0.000 ± 0.000,0.000 ± 0.000,0.000 ± 0.000


In [8]:
# External paired bootstrap: use the recomputed three-seed estimator from V13.
ci_out = ci.copy()
ci_out["Dataset"] = ci_out.dataset.map(DATASET_NAMES)
ci_out["Comparison"] = ci_out.model_A.map(MODEL_NAMES) + " − " + ci_out.model_B.map(MODEL_NAMES)
ci_out["Delta macro-F1 (95% CI)"] = ci_out.apply(
    lambda r: f"{r.delta_macro_f1:+.4f} ({r.ci_low_95:.4f}, {r.ci_high_95:.4f})", axis=1)
ci_out[["Dataset", "Comparison", "Delta macro-F1 (95% CI)", "bootstrap_n", "resampling", "estimator"]].to_csv(
    O / "Table3_External_PairedBootstrap_CI.csv", index=False, encoding="utf-8-sig")
print("External bootstrap: class-stratified paired-image resampling, 1,500 replicates.")
print("WBCBench film-cluster intervals are NOT reconstructed from these external records.")
display(ci_out[["Dataset", "Comparison", "Delta macro-F1 (95% CI)"]])

External bootstrap: class-stratified paired-image resampling, 1,500 replicates.
WBCBench film-cluster intervals are NOT reconstructed from these external records.


,Dataset,Comparison,Delta macro-F1 (95% CI)
0,PBC,ConvNeXt-Tiny − ResNet-50,"+0.0395 (0.0359, 0.0433)"
1,PBC,ConvNeXt-Tiny − RETFound,"+0.0916 (0.0862, 0.0970)"
2,PBC,ResNet-50 − RETFound,"+0.0521 (0.0464, 0.0574)"
3,Raabin-WBC,ConvNeXt-Tiny − ResNet-50,"+0.0867 (0.0770, 0.0967)"
4,Raabin-WBC,ConvNeXt-Tiny − RETFound,"+0.2667 (0.2554, 0.2784)"
5,Raabin-WBC,ResNet-50 − RETFound,"+0.1800 (0.1691, 0.1910)"


In [9]:
# External BNE+SNE mapping sensitivity (source: verified V13 summary).
sensitivity = external.copy()
sensitivity["Dataset"] = sensitivity.dataset.map(DATASET_NAMES)
sensitivity["Model"] = sensitivity.model.map(MODEL_NAMES)
sensitivity[["Dataset", "Model", "n_images", "baseline_mean", "baseline_sample_sd",
             "bne_sne_mean", "bne_sne_sample_sd", "delta_mean", "delta_sample_sd",
             "neutrophil_recall_baseline_mean", "neutrophil_recall_bne_sne_mean"]].to_csv(
    O / "Table_External_BNE_SNE_Sensitivity.csv", index=False, encoding="utf-8-sig")

In [10]:
# Figure 1 — study workflow (schematic; no model inference).
fig, ax = plt.subplots(figsize=(15, 3.7))
ax.set(xlim=(0, 1), ylim=(0, 1))
ax.axis("off")
items = [
    (0.012, "WBCBench\n13 classes\ntrain / validation / test"),
    (0.215, "3 configurations\n3 random seeds\nvalidation-only selection"),
    (0.418, "Locked WBCBench test\n13-class metrics\nclass-wise analysis"),
    (0.621, "External PBC / Raabin\n5 shared classes\nno target adaptation"),
    (0.824, "Uncertainty and checks\npaired image bootstrap\nBNE+SNE sensitivity"),
]
width = 0.166
for x, label in items:
    rect = FancyBboxPatch((x,0.26), width, 0.52, boxstyle="round,pad=0.025",
                          transform=ax.transAxes, facecolor="white", edgecolor="black", linewidth=1.1)
    ax.add_patch(rect)
    ax.text(x+width/2,0.52,label,ha="center",va="center",fontsize=9.5,transform=ax.transAxes)
for i in range(len(items)-1):
    x1=items[i][0]+width+0.004; x2=items[i+1][0]-0.005
    ax.annotate("",xy=(x2,0.52),xytext=(x1,0.52),xycoords=ax.transAxes,
                arrowprops={"arrowstyle":"->","lw":1.3})
ax.set_title("Study design and evaluation workflow",fontsize=13)
fig.savefig(O / "Figure1_Study_Workflow.png",dpi=300,bbox_inches="tight")
plt.close(fig)

In [11]:
# Figure 2 — corrected three-dataset Macro-F1, each model evaluated at three seeds.
fig, ax = plt.subplots(figsize=(9.2,5.3))
x=np.arange(len(DATASETS)); width=0.245
for j,m in enumerate(MODELS):
    q=agg[agg.model==m].set_index("dataset").loc[list(DATASETS)]
    ax.bar(x+(j-1)*width,q.macro_f1_mean.to_numpy(),width,
           yerr=q.macro_f1_sd.to_numpy(),capsize=3,label=MODEL_NAMES[m])
ax.set_xticks(x, [DATASET_NAMES[d] for d in DATASETS]); ax.set_ylabel("Macro-F1")
ax.set_ylim(0,1.05);ax.legend(frameon=False,loc="upper right")
ax.set_title("WBCBench and adaptation-free external performance (three-seed mean ± SD)")
ax.text(0.5,-0.15,"WBCBench: 13 classes; external datasets: 5 classes (not directly comparable)",
        transform=ax.transAxes,ha="center",va="center",fontsize=9)
fig.tight_layout()
fig.savefig(O / "Figure2_MacroF1_Three_Datasets.png",dpi=300,bbox_inches="tight")
plt.close(fig)

In [12]:
# Figure 3 — corrected BNE+SNE sensitivity, NOT superseded confusion matrices.
fig, axs = plt.subplots(1, 2, figsize=(11.3,4.0),sharey=True)
for ax, d in zip(axs, DATASETS[1:]):
    sub = external[external.dataset==d].set_index("model").loc[list(MODELS)]
    delta = sub.delta_mean.to_numpy()
    ax.bar(range(3),delta)
    ax.axhline(0,color="black",linewidth=0.85)
    ax.set_xticks(range(3),[MODEL_NAMES[m] for m in MODELS],rotation=17,ha="right")
    ax.set_title(DATASET_NAMES[d])
    for i,val in enumerate(delta):
        ax.annotate(f"{val:+.4f}",(i,val),xytext=(0,5 if val>=0 else -7),
                    textcoords="offset points",ha="center",va="bottom" if val>=0 else "top",fontsize=9)
axs[0].set_ylabel("Δ Macro-F1 (BNE+SNE − baseline)")
lo = float(min(external.delta_mean.min(),0)); hi=float(max(external.delta_mean.max(),0))
axs[0].set_ylim(lo-0.003,hi+0.002)
fig.suptitle("Sensitivity to combining BNE and SNE probabilities (three-seed mean)")
fig.tight_layout()
fig.savefig(O / "Figure3_External_BNE_SNE_Sensitivity.png",dpi=300,bbox_inches="tight")
plt.close(fig)

In [13]:
# Figure 4 — optional: reuse existing WBCBench XAI images only, without new model calls.
if EXPORT_XAI_FIGURE4:
    xdir=R / "outputs" / "asama14b_xai"
    images=sorted(xdir.rglob("*.png")) if xdir.is_dir() else []
    terms=("all_models_correct", "only_retfound_fullft_correct", "MMY_errors")
    chosen=[]
    for term in terms:
        matches=[p for p in images if term.lower() in str(p).lower()]
        if matches:chosen.append(matches[0])
    if len(chosen)==len(terms):
        fig, axes=plt.subplots(len(chosen),1,figsize=(13,4*len(chosen)))
        for ax,p in zip(np.atleast_1d(axes),chosen):
            ax.imshow(plt.imread(p));ax.axis("off");ax.set_title(p.stem)
        fig.suptitle("WBCBench qualitative attribution examples (previously generated)")
        fig.tight_layout()
        fig.savefig(O / "Figure4_Existing_WBCBench_XAI.png",dpi=300,bbox_inches="tight")
        plt.close(fig)
        print("Figure 4 reconstructed from the original WBCBench XAI exports.")
    else:
        print("Figure 4 not generated: fewer than three requested WBCBench XAI cases found.")
else:
    print("Figure 4 intentionally unchanged; use the previously verified manuscript image.")

Figure 4 reconstructed from the original WBCBench XAI exports.


In [14]:
# Machine-readable summary (no local file paths or user identifiers are written into the export).
summary = {
    "source": "verified_V13_external_and_retained_WBCBench_predictions",
    "external_verified_runs": 18,
    "wbcbench_checked_runs": 9,
    "external_bootstrap_replicates": 1500,
    "external_ci_resampling_unit": "class_stratified_paired_image",
    "external_ci_estimator": "three_seed_mean_macro_f1_difference",
    "note": "External RETFound results use corrected global-pooling checkpoint evaluation. WBCBench V12 replay is not exact; consult its audit for the nine changed predictions.",
    "output_files": sorted(f.name for f in O.iterdir() if f.is_file())
}
(O / "source_summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
print("Exported files:")
for file in sorted(O.iterdir()):
    if file.is_file(): print(" -",file.name)
print("Use the revised manuscript's Table/Figure numbers when inserting these outputs.")

Exported files:
 - Figure1_Study_Workflow.png
 - Figure2_MacroF1_Three_Datasets.png
 - Figure3_External_BNE_SNE_Sensitivity.png
 - Figure4_Existing_WBCBench_XAI.png
 - source_summary.json
 - Table1_Main_Performance.csv
 - Table1_PerSeed_Performance.csv
 - Table1_ThreeSeed_Performance_Numeric.csv
 - Table2_WBCBench_PerClass_F1.csv
 - Table3_External_PairedBootstrap_CI.csv
 - Table_External_BNE_SNE_Sensitivity.csv
Use the revised manuscript's Table/Figure numbers when inserting these outputs.
